# First contact: loading and inspecting a dataset

_Data Wrangling_

**Before you clean or model anything, look. Run the same inspection battery every single time.**

A dataset is a stranger. Before you do anything with it, you introduce yourself: you ask its size,
       its shape, and a handful of questions about every column. This takes about a minute and saves hours.

       The goal is a mental model of each column built from four facts:

---

This notebook is a practice scaffold for the **First contact: loading and inspecting a dataset** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — pandas

In [ ]:
import pandas as pd

# --- Load a real-ish dataset with mixed dtypes and real missingness ---
# Titanic ships with seaborn; or use any titanic.csv.
import seaborn as sns
df = sns.load_dataset('titanic')      # 891 rows, mixed numeric / categorical / bool

# === 1. SHAPE — is it the size I expect? ===
print(df.shape)                       # -> (891, 15)   rows x columns
# Look for: wildly wrong row/column counts = a broken load.

# === 2. EYEBALL ROWS — head / tail / sample ===
print(df.head())                      # opening rows
print(df.tail())                      # end rows — watch for footer/total junk
print(df.sample(5, random_state=0))   # a fair middle slice (head is often special)
# Look for: shifted columns, stray text, obviously wrong values.

# === 3. INFO — dtype + NON-NULL count per column (the densest command) ===
df.info()
# Look for: a non-null count below 891 = missing values in that column;
#           a dtype of 'object' on something that should be numeric.

# === 4. DTYPES — types on their own ===
print(df.dtypes)
# Look for: numbers stored as strings ('object' where you expect int/float).

# === 5. DESCRIBE — numeric AND categorical (don't skip the second call!) ===
print(df.describe())                       # count/mean/std/min/quartiles/max (numeric)
print(df.describe(include='object'))       # count/unique/top/freq (categoricals)
# Plain .describe() IGNORES non-numeric columns — include='object' covers them.

# === 6. NUNIQUE — cardinality (distinct values per column) ===
print(df.nunique().sort_values(ascending=False))
# Look for: 1 = constant (drop); 2 = flag/label; ~n_rows = an ID;
#           a small number on a 'numeric' column = secretly categorical.

# === 7. ISNA — exact missing-value count per column, worst first ===
print(df.isna().sum().sort_values(ascending=False))
# Titanic: 'deck' ~688 missing, 'age' ~177 missing, 'embarked'/'embark_town' a few.

# === 8. MEMORY — bytes per column (deep=True counts object strings honestly) ===
print(df.memory_usage(deep=True).sort_values(ascending=False))
# Look for: fat 'object' columns — candidates for the lighter 'category' dtype.

# Only AFTER all eight do you start cleaning. You now know every column's
# type, range, missingness, and cardinality.

## Visualize the data & results

_On first contact with load_breast_cancer, what does the cardinality (number of distinct values) of each column tell us at a glance — and which column is obviously the label?_

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

# Real bundled dataset: 569 cell-nucleus measurements, 30 features + target.
d = load_breast_cancer(as_frame=True)
df = d.frame
print('shape:', df.shape)                 # -> (569, 31)

# A representative subset of columns for the chart.
cols = ['mean radius', 'mean texture', 'mean perimeter', 'mean area',
        'mean smoothness', 'mean concavity', 'worst area', 'target']

# CARDINALITY = number of distinct values per column (.nunique()).
card = df[cols].nunique()
print(card)
# mean radius      456     mean concavity   537
# mean texture     479     worst area       544
# mean perimeter   522     target             2   <- only 2 values => the LABEL
# mean area        539
# mean smoothness  474
# The seven feature columns have hundreds of distinct values (continuous);
# 'target' has exactly 2 -> it is the binary class label, visible at a glance.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** You load a CSV and call df.describe(). It shows 6 numeric columns looking fine, so you start modeling. The file actually had 12 columns. What did .describe() hide, and what two commands should you have run instead?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall that plain .describe() summarizes only numeric columns by default. — _The other 6 columns are non-numeric (object/category) and were silently skipped &mdash; you never saw them._
- Run df.info() to list all 12 columns with their dtypes and non-null counts. — _It reveals the categorical columns and any missingness in one table, so nothing is invisible._
- Run df.describe(include='object') (or include='all') to summarize the categoricals. — _This gives count, unique, top, and freq for the text columns that plain .describe() ignored._

**Answer:** Plain .describe() hid the 6 non-numeric columns entirely &mdash; their existence, their missingness, and their distributions. Run df.info() (all columns: dtype + non-null count) and df.describe(include='object') (or include='all') so categoricals are summarized too. Never treat numeric .describe() as a full picture.

</details>

**Problem 2.** df.dtypes shows a column price as object instead of a float. df['price'].mean() errors. What is the likely cause, and how do you confirm and fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recognize that an object dtype on a "number" column means the values are stored as strings. — _Pandas falls back to object when even one entry can't be parsed as a number._
- Eyeball the values with df['price'].sample(10) or df['price'].unique()[:20]. — _You will likely spot culprits like "$5.00", "1,234", or "N/A" that force the whole column to text._
- Clean and cast: strip the symbols, then pd.to_numeric(df['price'], errors='coerce'). — _errors='coerce' turns unparseable entries into NaN so the column becomes a true numeric dtype._

**Answer:** The column holds numbers stored as strings &mdash; a stray "$",
        ",", or "N/A" poisoned it to object. Confirm by sampling the
        raw values, then clean the symbols and run pd.to_numeric(..., errors='coerce') to cast
        it. This is exactly why you check .dtypes on every load.

</details>

**Problem 3.** df.nunique() returns: user_id: 50000 (out of 50000 rows), is_churned: 2, state: 50, visits: 90. Classify each column from its cardinality alone.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare each column's unique count to the row count (50000). — _Cardinality relative to n_rows is what distinguishes IDs, flags, categories, and continuous features._
- user_id = 50000 unique = one value per row. — _That is an identifier; it must be dropped as a feature or it leaks/overfits._
- is_churned = 2, state = 50 (moderate), visits = 90 (many but bounded). — _2 is a binary label/flag; 50 is a clean categorical; ~90 is a genuine numeric/count feature._

**Answer:** user_id (50000 unique = n_rows) is an ID &mdash; drop it.
        is_churned (2) is a binary label/flag. state (50) is a true
        categorical. visits (90, bounded) is a numeric/count feature. Cardinality
        alone tells you the role of each column &mdash; which is why .nunique() is in the
        battery.

</details>